# Implementing A Six-Compartment Model for COVID-19 with Transmission Dynamics and Public Health Strategies


## About the Paper
Implementing paper titled **A six-compartment model for COVID-19 with transmission dynamics and public health strategies** written byt
Venkatesh Ambalarajan, Ankamma Rao Mallela, Vinoth Sivakumar, Prasantha Bharathi Dhandapani, Víctor Leiva, Carlos Martin-Barreiro & Cecilia Castro with [DOI](https://doi.org/10.1038/s41598-024-72487-9). 

### 1. Defining the Model Parameters

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp 

In [2]:
# Parameters from Table 1
Lambda = 0  # Assuming no recruitment for simplicity
mu = 0.00004  # Natural mortality rate
theta = 0.5  # Proportion of asymptomatic cases reported as symptomatic
omega = 0.2  # Conversion rate from asymptomatic to symptomatic per day
zeta_A = 0.4  # Adjustment parameter for asymptomatic
zeta_I = 0.3  # Adjustment parameter for reported symptomatic
zeta_G = 0.3  # Adjustment parameter for isolated
beta = 0.4162  # Transmission rate per day
eta_I = 0.0175  # Isolation rate from reported symptomatic per day
eta_U = 0.0768  # Isolation rate from unreported symptomatic per day
gamma_I = 0.0714  # Recovery rate from reported symptomatic per day
gamma_U = 0.0714  # Recovery rate from unreported symptomatic per day
gamma_G = 0.0714  # Recovery rate from isolated per day
mu_I = 0.0016  # Mortality rate of reported symptomatic per day
mu_U = 0.0016  # Mortality rate of unreported symptomatic per day
mu_G = 0.0025  # Mortality rate of isolated per day

### Implementing The Differential Equation Method

In [4]:
def dSdt(S, A, I, U, G, R):
    return Lambda - beta * (zeta_A * A + zeta_I * I + zeta_G * G) * S / N - mu * S


def dAdt(S, A, I, U, G, R):
    return beta * (zeta_A * A + zeta_I * I + zeta_G * G) * S / N - (omega + mu) * A


def dIdt(A, I, U, G, R):
    return theta * omega * A - (eta_I + gamma_I + mu_I + mu) * I


def dUdt(A, I, U, G, R):
    return (1 - theta) * omega * A - (eta_U + gamma_U + mu_U + mu) * U


def dGdt(I, U, G, R):
    return eta_I * I + eta_U * U - (gamma_G + mu_G + mu) * G


def dRdt(I, U, G, R):
    return gamma_I * I + gamma_U * U + gamma_G * G - mu * R


# Initial conditions
S0 = 1e6  # Initial susceptible population
A0 = 100  # Initial asymptomatic cases
I0 = 50   # Initial reported symptomatic cases
U0 = 50   # Initial unreported symptomatic cases
G0 = 0    # Initial isolated cases
R0 = 0    # Initial recovered cases
N = S0 + A0 + I0 + U0 + G0 + R0  # Total population

# Time span for simulation
t_span = (0, 400)
t = np.linspace(t_span[0], t_span[1], 401)

# Define the system of ODEs


def model(t, y):
    S, A, I, U, G, R = y
    dS = dSdt(S, A, I, U, G, R)
    dA = dAdt(S, A, I, U, G, R)
    dI = dIdt(A, I, U, G, R)
    dU = dUdt(A, I, U, G, R)
    dG = dGdt(I, U, G, R)
    dR = dRdt(I, U, G, R)
    return [dS, dA, dI, dU, dG, dR]

# Solve the ODEs
sol = solve_ivp(model, t_span, [S0, A0, I0, U0, G0, R0], t_eval=t)